
<a id="top"></a>
# Modulo 4 — Marketing bancario, pipeline e tracciamento con MLflow

## Indice

- [Contesto: perché parlare di MLOps](#contesto-mlops)
- [Setup: librerie, esperimento MLflow e dati grezzi](#setup)
- [Preparazione del dataset e train/test split](#prep-split)
- [Pipeline di feature engineering con feature-engine](#pipeline-feature-engine)
- [Addestramento e confronto di più modelli](#confronto-modelli)
- [Lettura degli esperimenti dalla UI di MLflow](#mlflow-ui)
- [Esercizio guidato](#esercizio)
- [Dal modello tracciato all’inferenza riutilizzabile](#modello-inferenza)



<a id="contesto-mlops"></a>
## Contesto: perché parlare di MLOps

Finora il lavoro è rimasto dentro al notebook: lettura dei dati, preparazione, scelta di un modello e prima validazione.  
Funziona, ma è fragile. Piccole differenze di ambiente, trasformazioni fatte a mano e modelli non tracciati rendono difficile riprodurre i risultati o consegnare il lavoro a un team di ingegneri.

Qui entra in gioco il punto di vista **MLOps**. L'idea è trattare il ciclo di vita del modello in modo più simile al software:

- passaggi chiari e ripetibili;
- esperimenti tracciati;
- modelli salvati in un formato standard e riutilizzabile.

In questo notebook useremo **MLflow** per tracciare gli esperimenti e lo standardizzare il salvataggio dei modelli, e **feature-engine** per costruire pipeline di preprocessing leggibili e riutilizzabili.  
Il dataset resta quello del **marketing bancario**, ma l'attenzione si sposta dal *cosa* modelliamo a *come* lo facciamo in modo strutturato.

<div style="background-color:#f5f5f5; padding:10px; border-radius:4px">
<b>Nota importante</b><br>
Un modello addestrato senza tracciamento e senza pipeline resta un prototipo.  
Un modello inserito in una pipeline, tracciato con MLflow e salvato in un formato standard comincia ad assomigliare a qualcosa che può vivere fuori dal notebook.
</div>



<a id="setup"></a>
## Setup: librerie, esperimento MLflow e dati grezzi

Per prima cosa importiamo le librerie, configuriamo un esperimento MLflow dedicato a questo modulo e carichiamo i dati grezzi del marketing bancario dalla libreria `ucimlrepo`.

L'esperimento MLflow ci permetterà di raccogliere in un unico contenitore tutte le esecuzioni di training che facciamo qui.


In [ ]:

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from feature_engine.imputation import MeanMedianImputer
from feature_engine.encoding import OneHotEncoder as FEOneHotEncoder
from feature_engine.wrappers import SklearnTransformerWrapper
from feature_engine.pipeline import Pipeline as FEPipeline

# Caricamento dataset Bank Marketing dalla UCI
bank_marketing = fetch_ucirepo(id=222)

X_raw = bank_marketing.data.features.copy()
y_raw = bank_marketing.data.targets.copy()
# Sample di 10kcrighe per velocizzare l'esecuzione
X_raw, _, y_raw, _ = train_test_split(X_raw, y_raw, train_size=10000, random_state=42, stratify=y_raw)

print("Shape originale X:", X_raw.shape)
print("Shape originale y:", y_raw.shape)
print("Prime colonne:", X_raw.columns.tolist()[:10])

display(X_raw.head())
display(y_raw.head())



<a id="prep-split"></a>
## Preparazione del dataset e train/test split

In questa sezione applichiamo solo trasformazioni **deterministiche**, che non richiedono stime dalla distribuzione dei dati.  
Tutto ciò che deve essere `fit` (imputazione, scaling, encoding) verrà spostato dentro la pipeline di `feature-engine`.

Rimodelliamo il dataset in modo coerente con quanto fatto nei moduli precedenti:

- mappiamo il target `y` da `yes`/`no` a 1/0;
- convertiamo alcune variabili testuali in binarie 0/1;
- creiamo poche variabili derivate utili per la campagna;
- rimuoviamo la variabile `duration`, che contiene informazione post-chiamata e non è disponibile in fase di pianificazione.


In [ ]:

# Costruiamo un DataFrame unico con feature e target
df = X_raw.copy()
target_col = y_raw.columns[0]
df[target_col] = y_raw[target_col]

print("Colonna target:", target_col)
print("Shape iniziale df:", df.shape)

# Mappatura del target y: yes/no -> 1/0
df[target_col] = df[target_col].map({"yes": 1, "no": 0})

# Alcune variabili binarie: default, housing, loan
binary_yes_no_cols = [col for col in ["default", "housing", "loan"] if col in df.columns]

for col in binary_yes_no_cols:
    # "unknown" viene trasformato in NaN, poi il mapping in 0/1
    df[col] = df[col].replace("unknown", np.nan)
    df[col] = df[col].map({"yes": 1, "no": 0})

# Variabile: cliente è mai stato contattato prima? (pdays)
if "pdays" in df.columns:
    df["was_previously_contacted"] = np.where(df["pdays"] != -1, 1, 0)
    df["pdays"] = df["pdays"].replace(-1, np.nan)


# Education come variabile ordinale
education_mapping = {
    "primary": 1,
    "secondary": 2,
    "tertiary": 3,

}
df["education"] = df["education"].map(education_mapping)

# Rimozione della variabile post-chiamata 'duration' dalle feature
if "duration" in df.columns:
    df = df.drop(columns=["duration"])
    print("Colonna 'duration' rimossa dalle feature.")

print("Shape df dopo le trasformazioni deterministiche:", df.shape)

# Separazione X e y
y = df[target_col]
X = df.drop(columns=[target_col])

print("Shape X:", X.shape)
print("Shape y:", y.shape)

# Train/test split con stratificazione
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

print("Shape X_train:", X_train.shape)
print("Shape X_test :", X_test.shape)



<a id="pipeline-feature-engine"></a>
## Pipeline di feature engineering con feature-engine

Per rendere il flusso ripetibile prepariamo i dati con una pipeline unica che si occupa di:

- imputare le variabili numeriche;
- codificare le variabili categoriche in dummies;
- scalare le variabili numeriche non binarie;
- applicare il modello finale.

Useremo la `Pipeline` di `feature-engine` per concatenare gli step e alcune classi dedicate:

- `MeanMedianImputer` per l'imputazione numerica;
- `OneHotEncoder` per l'encoding one-hot delle categoriche;
- `SklearnTransformerWrapper` per applicare `StandardScaler` solo ad alcune colonne.

Documentazione utile:

- Pipeline di feature-engine: https://feature-engine.trainindata.com/en/latest/pipeline/Pipeline.html  
- MeanMedianImputer: https://feature-engine.trainindata.com/en/latest/imputation/MeanMedianImputer.html  
- OneHotEncoder: https://feature-engine.trainindata.com/en/latest/encoding/OneHotEncoder.html  
- SklearnTransformerWrapper: https://feature-engine.trainindata.com/en/latest/wrappers/SklearnTransformerWrapper.html

<div style="background-color:#ffe9e9; padding:10px; border-radius:4px">
<b>Attenzione</b><br>
Tutta la logica di preprocessing deve stare nella pipeline. In questo modo lo stesso oggetto verrà usato sia in addestramento sia in inferenza, riducendo il rischio di discrepanze tra notebook e produzione.
</div>


In [ ]:

# Identificazione delle colonne per tipo
# Categoriali: oggetti o stringhe
categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
# Numeriche: tutto il resto
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
# Colonne binarie (0/1) tra le numeriche
binary_cols = []
for col in numeric_cols:
    vals = pd.Series(X_train[col].dropna().unique())
    if vals.nunique() <= 2 and set(vals.unique()).issubset({0, 1}):
        binary_cols.append(col)

# Colonne numeriche da scalare: numeriche non binarie
numeric_to_scale = [col for col in numeric_cols if col not in binary_cols]

print("Colonne categoriali:", categorical_cols)
print("Colonne numeriche   :", numeric_cols)
print("Colonne binarie     :", binary_cols)
print("Numeriche da scalare:", numeric_to_scale)


In [ ]:
df['contact']

In [ ]:
from sklearn.preprocessing import RobustScaler

from feature_engine.imputation import MeanMedianImputer, CategoricalImputer
from feature_engine.encoding import OneHotEncoder as FEOneHotEncoder
from feature_engine.wrappers import SklearnTransformerWrapper
from feature_engine.pipeline import Pipeline as FEPipeline

# colonne categoriche e numeriche
categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()

# numeriche da scalare: escludo le binarie
binary_cols = []
for col in numeric_cols:
    vals = X_train[col].dropna().unique()
    if len(vals) <= 2 and set(vals).issubset({0, 1}):
        binary_cols.append(col)

numeric_to_scale = [c for c in numeric_cols if c not in binary_cols]

# PIPELINE
pipeline = FEPipeline(
    steps=[
        # 1. Imputazione mediana numeriche
        (
            "median_num",
            MeanMedianImputer(
                imputation_method="median",
                variables=numeric_cols
            )
        ),

        # 2. Imputazione categoriche
        (
            "impute_cat",
            CategoricalImputer(
                imputation_method="missing",
                variables=categorical_cols
            )
        ),

        # 3. One-hot encoding
        (
            "one_hot",
            FEOneHotEncoder(
                variables=categorical_cols,
                drop_last=True,
                ignore_format=True
            )
        ),

        # 4. Robust scaler sulle numeriche non binarie
        (
            "robust_scaler",
            SklearnTransformerWrapper(
                transformer=RobustScaler(),
                variables=numeric_to_scale
            )
        ),
    ]
)

# fit + transform
pipeline.fit(X_train, y_train)
X_train_t = pipeline.transform(X_train)
X_test_t  = pipeline.transform(X_test)

X_train_t.head()

In [ ]:
# ============================
# IMPORT
# ============================

from feature_engine.imputation import (
    MeanMedianImputer,
    CategoricalImputer
)
from feature_engine.encoding import OneHotEncoder as FEOneHotEncoder
from feature_engine.wrappers import SklearnTransformerWrapper
from feature_engine.pipeline import Pipeline as FEPipeline

from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC


# ============================
# PIPELINE
# ============================

rf_pipeline = FEPipeline(steps=[
    ("median_num", MeanMedianImputer(
        imputation_method="median",
        variables=numeric_cols
    )),
    
    ("impute_cat", CategoricalImputer(
        imputation_method="missing",
        variables=categorical_cols
    )),
    
    ("one_hot", FEOneHotEncoder(
        drop_last=True,
        ignore_format=True,
        variables=categorical_cols
    )),

    ("model", RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ))
], verbose=True)


lr_pipeline = FEPipeline(steps=[
    ("median_num", MeanMedianImputer(
        imputation_method="median",
        variables=numeric_cols
    )),
    
    ("impute_cat", CategoricalImputer(
        imputation_method="missing",
        variables=categorical_cols
    )),
    
    ("one_hot", FEOneHotEncoder(
        drop_last=True,
        ignore_format=True,
        variables=categorical_cols
    )),
    
    ("robust_scaler", SklearnTransformerWrapper(
        RobustScaler(),
        variables=numeric_to_scale
    )),

    ("model", LogisticRegression(
        random_state=42
    ))
], verbose=True)


svm_pipeline = FEPipeline(steps=[
    ("median_num", MeanMedianImputer(
        imputation_method="median",
        variables=numeric_cols
    )),
    
    ("impute_cat", CategoricalImputer(
        imputation_method="missing",
        variables=categorical_cols
    )),
    
    ("one_hot", FEOneHotEncoder(
        drop_last=True,
        ignore_format=True,
        variables=categorical_cols
    )),
    
    ("robust_scaler", SklearnTransformerWrapper(
        RobustScaler(),
        variables=numeric_to_scale
    )),
         
    ("model", SVC(
        random_state=42
    ))
], verbose=True)


<a id="confronto-modelli"></a>
## Addestramento e confronto di più modelli

Ora definiamo tre modelli da confrontare, utilizzando sempre la stessa pipeline di preprocessing:

- **RandomForestClassifier**: modello ad alberi, non lineare, spesso robusto su dati tabellari.
- **LogisticRegression con penalty elastic-net**: modello lineare con regolarizzazione mista L1/L2, utile per controllare la complessità.
- **SVM con kernel RBF (SVC)**: modello di margine massimo che lavora tramite una funzione kernel; useremo il valore di `decision_function` per calcolare l'AUC.

Ogni modello viene inserito in una pipeline `feature-engine` costruita con `make_preprocessing_pipeline`.  
Ogni addestramento viene eseguito dentro un `mlflow.start_run`, in modo che parametri, metriche e modello siano salvati automaticamente nell'esperimento configurato.


In [ ]:
import mlflow
import mlflow.sklearn

def train_model(model_name, pipeline, X_train, y_train, X_test, y_test):

    with mlflow.start_run(run_name=model_name):
        # Fit del modello (preprocessing + classificatore)
        print("Training del modello:", model_name)
        pipeline.fit(X_train, y_train)
        print("Modello addestrato.")
        # Predizioni sul test set
        y_pred = pipeline.predict(X_test)

        # Score continuo per ROC AUC
        if hasattr(pipeline, "predict_proba"):
            y_score = pipeline.predict_proba(X_test)[:, 1]
        else:
            # per SVM usiamo la decision_function
            y_score = pipeline.decision_function(X_test)
        # Metriche principali sul test set
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_score)

        # Logging esplicito su MLflow (in aggiunta all'autolog)
        mlflow.log_metric("accuracy_test", acc)
        mlflow.log_metric("precision_test", prec)
        mlflow.log_metric("recall_test", rec)
        mlflow.log_metric("f1_test", f1)
        mlflow.log_metric("roc_auc_test", auc)

        # Tag contestuali utili
        mlflow.set_tag("dataset", "bank_marketing")
        mlflow.set_tag("problem_type", "binary_classification")

        # ✓ LOG DELLA PIPELINE COMPLETA (preprocessing + modello)
        # Questo salva l'intera pipeline come artifact riutilizzabile
        mlflow.sklearn.log_model(
            pipeline,
            artifact_path="pipeline",
            input_example=X_test.head(5)
        )
        print(f"✓ Pipeline completa loggata in MLflow per {model_name}")


In [ ]:
import os

# ============================
# CONFIGURAZIONE MLflow - SALVA NELLA ROOT
# ============================
# Cambia la directory nella root del progetto per salvare mlruns lì
root_dir = os.path.abspath("../")  # Torna alla root (3.0)
os.chdir(root_dir)

# Configura MLflow per salvare i dati nella root
mlflow.set_tracking_uri(f"file:{os.path.join(root_dir, 'mlruns')}")

# Ora configura l'esperimento
mlflow.set_experiment("bank_marketing_modulo4_mlops")
mlflow.sklearn.autolog()

print(f"✓ MLflow configurato per salvare in: {os.path.join(root_dir, 'mlruns')}")
print(f"✓ Directory di lavoro: {os.getcwd()}")

In [ ]:
train_model("Random_Forest", rf_pipeline, X_train, y_train, X_test, y_test)

In [ ]:
train_model("Logistic_Regression", lr_pipeline, X_train, y_train, X_test, y_test)

In [ ]:
train_model("SVC", svm_pipeline, X_train, y_train, X_test, y_test)


<a id="mlflow-ui"></a>
## Lettura degli esperimenti dalla UI di MLflow

Gli esperimenti registrati da questo notebook possono essere ispezionati in modo più comodo tramite l'interfaccia web di MLflow.

Dal terminale, nella cartella del progetto, si può eseguire:

```bash
mlflow ui --port 5000
```

e aprire il browser all'indirizzo:

```text
http://127.0.0.1:5000
```

Dentro la UI troverai:

- l'esperimento `bank_marketing_modulo4_mlops`;
- le run per ciascun modello, con le metriche che abbiamo loggato sul test set;
- i parametri dei modelli, l'ambiente software e il modello salvato in formato MLflow.

Questa schermata diventa il punto di riferimento per:

- confrontare i risultati dei diversi tentativi;
- ritrovare a distanza di tempo il modello effettivamente scelto;
- capire quali impostazioni hanno portato a una certa performance.



<a id="esercizio"></a>
## Esercizio guidato

<div style="background-color:#f0f7ff; padding:10px; border-radius:4px">
<b>Esercizio</b><br>
L'obiettivo è usare la stessa struttura del notebook per esplorare l'effetto degli iperparametri sui risultati tracciati in MLflow.
<ol>
<li>Nel blocco in cui definiamo il dizionario <code>models</code>, crea una variante di <code>RandomForest</code> con meno alberi (ad esempio 100 stimatori) e un <code>max_depth</code> limitato.</li>
<li>Rilancia il blocco di addestramento e confronta le nuove run nella UI di MLflow: osserva come cambiano in particolare <code>roc_auc_test</code> e <code>recall_test</code>.</li>
<li>Modifica il parametro <code>C</code> del modello SVM (ad esempio 0.1 e 10.0), ripeti l'addestramento e analizza come varia il compromesso tra overfitting e underfitting nei log di MLflow.</li>
</ol>
Prova a rispondere, per ogni variante, a questa domanda: <i>in un contesto di campagna marketing, quale configurazione ti sembra più adatta e perché?</i>
</div>


<a id="modello-inferenza"></a>
## Dal modello tracciato all'inferenza riutilizzabile

Uno dei vantaggi principali di MLflow è la possibilità di ricaricare un modello salvato all'interno di un'altra applicazione, con lo stesso comportamento visto nel notebook.

**Cosa logghiamo adesso:**

- L'intera **pipeline di feature-engine** (non solo il modello), che contiene:
  - Imputazione numerica (mediana)
  - Imputazione categorica
  - One-hot encoding
  - Scaling robusto
  - Modello di classificazione (RF, LR, SVM)

- Quando carichi la pipeline da MLflow, ottieni un oggetto ready-to-use che puoi usare per l'inferenza direttamente in produzione

**Come funziona:**

1. Nella cella seguente, usiamo l'API di MLflow (`MlflowClient`) per cercare **automaticamente** la run con il miglior AUC
2. Carichiamo la pipeline da quella run
3. Usiamo la pipeline per fare previsioni su nuovi dati

Non devi più copiare manualmente il `run_id`!


In [ ]:
# ============================
# RECUPERO AUTOMATICO DELLA MIGLIORE RUN
# ============================

from mlflow.tracking import MlflowClient

client = MlflowClient()

# Ottieni l'experiment_id
experiment = client.get_experiment_by_name("bank_marketing_modulo4_mlops")
experiment_id = experiment.experiment_id

# Cerca tutte le run ordinate per AUC (migliore prima)
runs = client.search_runs(
    experiment_ids=[experiment_id],
    order_by=["metrics.roc_auc_test DESC"],
    max_results=10
)

if runs:
    best_run = runs[0]
    best_run_id = best_run.info.run_id
    best_auc = best_run.data.metrics.get("roc_auc_test", "N/A")
    
    print(f"✓ Migliore run trovata:")
    print(f"  Run ID: {best_run_id}")
    print(f"  Model: {best_run.info.run_name}")
    print(f"  AUC: {best_auc:.4f}")
else:
    print("❌ Nessuna run trovata!")
    best_run_id = None

# ============================
# CARICAMENTO DELLA PIPELINE COMPLETA
# ============================

if best_run_id:
    logged_model_uri = f"runs:/{best_run_id}/pipeline"
    
    try:
        loaded_pipeline = mlflow.sklearn.load_model(logged_model_uri)
        print(f"\n✓ Pipeline caricata con successo!")
        print(f"  URI: {logged_model_uri}")
        
        # Fai previsioni su nuovi dati
        sample = X_test.head(10)
        preds = loaded_pipeline.predict(sample)
        proba = loaded_pipeline.predict_proba(sample)[:, 1]
        
        # Mostra i risultati
        results = pd.DataFrame({
            "y_actual": y_test.head(10).values,
            "y_pred": preds,
            "p_conversion": np.round(proba, 3)
        })
        
        print("\n✓ Previsioni sulla pipeline caricata:")
        display(results)
        
    except Exception as e:
        print(f"❌ Errore nel caricamento della pipeline: {e}")
        print(f"   Verifica che il run_id '{best_run_id}' esista e contenga il modello")



L'oggetto caricato da MLflow contiene:

- la pipeline di preprocessing definita con feature-engine;
- il modello finale addestrato;
- le informazioni di ambiente necessarie per riprodurre il contesto (versioni delle librerie, dipendenze).

È il punto di contatto naturale tra il lavoro nel notebook e un sistema di inferenza da mettere in produzione.
